# The clock: when do problems and stops happen?

**A2.** At what times of day and days of the week do noise complaints, assaults, and collisions peak?

**A6.** How do the timing patterns of calls for service and police stops differ across types of calls and stops?

**A7.** Do traffic stops happen at the same times as collisions, and does this differ between weekdays and weekends?

**Inputs:** `data/processed/calls_2021_2025.csv.gz`, `data/processed/crime_cases_2021_2025.csv.gz`, `data/processed/collisions.csv`, `data/processed/vehicle_stops.csv`

**Outputs:** `figures/03_*.png` and `results/03_*.csv`

In [1]:
# Shared loaders, plot style, and project settings
import pandas as pd

from bpdtools.config import ANALYSIS_END, ANALYSIS_START, ASSAULT_INCIDENT_TYPES
from bpdtools.load import load_calls, load_cases
from bpdtools.paths import repo_path
from bpdtools.plots import set_plot_style
from bpdtools.traffic import load_collisions, load_stops

set_plot_style()

# Figures and result tables go to fixed folders at the repository root
figures = repo_path("figures")
results = repo_path("results")
figures.mkdir(exist_ok=True)
results.mkdir(exist_ok=True)

# Vehicle stops only start in 2024, so A7 uses the two years that stops
# and collisions both cover
STOP_YEARS_START = "2024-01-01"
STOP_YEARS_END = "2025-12-31"

## Load the data

All times are local Berkeley wall clock times.

In [2]:
def in_window(events, start, end):
    """Keep the events whose local time falls in a window of whole days.

    Parameters
    ----------
    events : pandas.DataFrame
        One row per event with a tz-naive `local_time` column.
    start, end : str
        First and last local date of the window, inclusive, as "YYYY-MM-DD".

    Returns
    -------
    pandas.DataFrame
        The rows of `events` from `start` 00:00 up to the end of `end`.
    """
    # The end date is inclusive, so stop before midnight of the next day
    after_end = pd.Timestamp(end) + pd.Timedelta(days=1)
    keep = (events["local_time"] >= pd.Timestamp(start)) & (
        events["local_time"] < after_end
    )
    return events.loc[keep]


# Calls and crime cases are already cut to 2021-2025 by their loaders
calls = load_calls()
cases = load_cases()

# Collisions go back to 2007, so cut them to the same 2021-2025 window
collisions = in_window(load_collisions(), ANALYSIS_START, ANALYSIS_END)

# One row per stop already (counted by LEA_Record_ID)
stops = load_stops()

# Assaults are crime cases of the two assault incident types
assaults = cases.loc[cases["Incident_Type"].isin(ASSAULT_INCIDENT_TYPES)]
noise_calls = calls.loc[calls["call_group"].eq("noise")]

sizes = pd.Series(
    {
        "calls": len(calls),
        "noise calls": len(noise_calls),
        "assault cases": len(assaults),
        "collisions 2021-2025": len(collisions),
        "vehicle stops": len(stops),
    },
    name="rows",
)
sizes

calls                   307628
noise calls              12693
assault cases             3567
collisions 2021-2025      4213
vehicle stops             7431
Name: rows, dtype: int64